# DINOv3 + YOLOv11 Object Detection Fine-tuning

This notebook fine-tunes a **YOLOv11** detection head on top of a **DINOv3 ViT-S/16** backbone,
using [Lightly Train](https://github.com/lightly-ai/lightly-train) for self-supervised
distillation and [Ultralytics](https://github.com/ultralytics/ultralytics) for supervised
fine-tuning and evaluation.

**Pipeline overview:**
1. Install dependencies
2. Prepare the dataset configuration (`data.yaml`)
3. Set training configuration
4. Validate the dataset
5. Distill DINOv3 into a YOLOv11 backbone (self-supervised pretraining)
6. Supervised fine-tuning with Ultralytics YOLO
7. Evaluate the fine-tuned model

> **Note:** Paths below (e.g. `/kaggle/input/...`, `/kaggle/working/...`) for Kaggle
> environment. Update `DATA_YAML` and `CONFIG["data_path"]` / `CONFIG["output_dir"]` if
> running elsewhere (e.g. locally or on Colab).

## 1. Install Required Libraries

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu1
!pip install -q lightly-train ultralytics opencv-python-headless matplotlib seaborn

## 2. Prepare Dataset Configuration

In [ ]:
import yaml

DATA_YAML = '/kaggle/input/graz-yolo-ready-v11/data.yaml'

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

# Update dataset path
data['path'] = '/kaggle/input/graz-yolo-ready-v11'

UPDATE_YAML = '/kaggle/working/data_updated.yaml'
# Save updated yaml
with open(UPDATE_YAML, 'w') as f:
    yaml.dump(data, f)

print(data)

## 3. Imports and Training Configuration

In [ ]:
import os, warnings
from pathlib import Path

import lightly_train as lt
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import yaml
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# Centralized Configuration
CONFIG = {
    # Paths
    "data_path": "/kaggle/input/graz-yolo-ready-v11",
    "output_dir": "/kaggle/working/dinov3_yolo_finetuned-ds-v1",

    # Model Setup
    "backbone": "dinov3/vits16",  # DINOv3 teacher model
    "model_arch": "yolo11n",      # YOLOv11 nano backbone

    # Training Hyperparameters
    "batch_size": 16,
    "epochs": 50,
    "learning_rate": 1e-4,
    "img_size": 640,
    "workers": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Augmentations
    "mosaic": 1.0,
    "mixup": 0.2,

    # --- RESUME & CHECKPOINT CONFIGURATION ---
    "resume_pretrain": True,   # Toggle to resume Lightly DINOv3 distillation
    "resume_finetune": True,   # Toggle to resume Ultralytics YOLO fine-tuning

    "pretrain_checkpoint_dir": "/kaggle/working/dinov3_yolo_finetuned-ds-v1/pretrain",
    "distilled_final_weights": "/kaggle/working/dinov3_yolo_finetuned-ds-v1/pretrain/exported_models/exported_last.pt",
    "yolo_resume_weights": "/kaggle/working/dinov3_yolo_finetuned-ds-v1/finetuned_model/weights/last.pt",
    # -----------------------------------------
}

print(f"Using device: {CONFIG['device']}")

# Create output directory hierarchy
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

# Path to data.yaml verification
data_yaml_path = Path(UPDATE_YAML)
if not data_yaml_path.exists():
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}")

## 4. Data Validation

In [ ]:
with open(data_yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

num_classes = data_yaml["nc"]
class_names = data_yaml["names"]
print(f"📊 Dataset has {num_classes} classes: {class_names}")

## 5. Distill DINOv3 into YOLOv11 (Pretraining Phase)

In [ ]:
print("🚀 Starting DINOv3 + YOLOv11 Distillation (Self-Supervised)...")

train_images_dir = Path(CONFIG["data_path"]) / "images" / "train"

# Lightly pretrain step using resume configuration from CONFIG
lt.pretrain(
    out=CONFIG["pretrain_checkpoint_dir"],
    data=str(train_images_dir),
    model=f"ultralytics/{CONFIG['model_arch']}.yaml",
    method="distillation",
    method_args={
        "teacher": CONFIG["backbone"],
    },
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["workers"],
    accelerator="gpu" if CONFIG["device"] == "cuda" else "cpu",
    devices=1,
    resume_interrupted=CONFIG["resume_pretrain"],  # Controlled via CONFIG
)

print(f"✅ Pretraining stage handled. Weights verified at {CONFIG['distilled_final_weights']}")

## 6. Supervised Fine-Tuning with Ultralytics YOLO

In [ ]:
print("🚀 Preparing Supervised Fine-Tuning...")

yolo_resume_path = Path(CONFIG["yolo_resume_weights"])
distilled_weights_path = Path(CONFIG["distilled_final_weights"])

# Determine if we are resuming an interrupted YOLO training run
if CONFIG["resume_finetune"] and yolo_resume_path.exists():
    print(f"🔄 Interrupted YOLO run found! Resuming fine-tuning from checkpoint: {yolo_resume_path}")
    model = YOLO(str(yolo_resume_path))

    # Resuming training inherits all previous hyperparameters automatically
    finetune_results = model.train(resume=True)

else:
    print("🆕 No valid checkpoint found or resume disabled. Initializing fresh fine-tuning from distilled weights...")
    if not distilled_weights_path.exists():
        raise FileNotFoundError(f"Could not locate backbone weights at {distilled_weights_path}. Ensure pretraining completed.")

    model = YOLO(str(distilled_weights_path))

    finetune_results = model.train(
        data=str(data_yaml_path),
        epochs=CONFIG["epochs"],
        batch=CONFIG["batch_size"],
        lr0=CONFIG["learning_rate"],
        imgsz=CONFIG["img_size"],
        device=0 if CONFIG["device"] == "cuda" else "cpu",
        workers=CONFIG["workers"],
        mosaic=CONFIG["mosaic"],
        mixup=CONFIG["mixup"],
        project=CONFIG["output_dir"],
        name="finetuned_model",
    )

print(f"✅ Fine-Tuning complete!")

## 7. Evaluation & Metrics

In [ ]:
best_model_path = Path(CONFIG["output_dir"]) / "finetuned_model" / "weights" / "best.pt"

if best_model_path.exists():
    val_model = YOLO(str(best_model_path))

    val_results = val_model.val(
        data=str(data_yaml_path), split="val", device=0 if CONFIG["device"] == "cuda" else "cpu"
    )
    print("\n📊 Validation Results:")
    print(f"   mAP50-95: {val_results.box.map:.4f}")
    print(f"   mAP50: {val_results.box.map50:.4f}")
    print(f"   Precision: {val_results.box.mp:.4f}")
    print(f"   Recall: {val_results.box.mr:.4f}")

    if hasattr(val_results, "confusion_matrix"):
        cm = val_results.confusion_matrix.matrix
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues")
        plt.title("Confusion Matrix - Validation")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.savefig(Path(CONFIG["output_dir"]) / "confusion_matrix_val.png")
        plt.close()
        print("✅ Saved validation confusion matrix")
else:
    print(f"⚠️ Best fine-tuned model not found at {best_model_path}")

print("\n🎉 Pipeline execution complete!")